In [3]:
!pip install catboost
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# **1. LOAD DATA**

In [4]:
train = pd.read_csv(r'/content/train.csv')
test  = pd.read_csv(r'/content/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")

Train shape: (77299, 11)
Test shape : (41778, 10)


# **2. FEATURE ENGINEERING**

In [5]:
def feature_engineering(df):
    df = df.copy()

    # Parse timestamp
    df['hour']         = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute']       = df['timestamp'].str.split(':').str[1].astype(int)
    df['time_minutes'] = df['hour'] * 60 + df['minute']

    # Cyclic time encoding
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['time_sin']   = np.sin(2 * np.pi * df['time_minutes'] / 1440)
    df['time_cos']   = np.cos(2 * np.pi * df['time_minutes'] / 1440)

    # Geohash prefix levels
    df['geo3'] = df['geohash'].str[:3]
    df['geo4'] = df['geohash'].str[:4]
    df['geo5'] = df['geohash'].str[:5]

    # Binary encodings
    df['LargeVehicles_bin'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['Landmarks_bin']     = (df['Landmarks'] == 'Yes').astype(int)

    # Ordinal road type
    road_order = {'Residential': 0, 'Street': 1, 'Highway': 2}
    df['RoadType_ord'] = df['RoadType'].map(road_order).fillna(-1).astype(int)

    # Ordinal weather
    weather_order = {'Snowy': 0, 'Rainy': 1, 'Foggy': 2, 'Sunny': 3}
    df['Weather_ord'] = df['Weather'].map(weather_order).fillna(-1).astype(int)

    # Temperature: fill missing with per-weather group median
    df['Temperature'] = df.groupby('Weather')['Temperature'].transform(
        lambda x: x.fillna(x.median())
    )
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())

    # Interaction features
    df['lanes_x_road']     = df['NumberofLanes'] * df['RoadType_ord']
    df['lanes_x_vehicles'] = df['NumberofLanes'] * df['LargeVehicles_bin']
    df['temp_x_weather']   = df['Temperature']   * df['Weather_ord']
    df['road_x_landmark']  = df['RoadType_ord']  * df['Landmarks_bin']

    # Peak hour flags
    df['is_morning_peak'] = ((df['hour'] >= 7)  & (df['hour'] <= 9)).astype(int)
    df['is_evening_peak'] = ((df['hour'] >= 17) & (df['hour'] <= 19)).astype(int)
    df['is_night']        = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)

    return df

train = feature_engineering(train)
test  = feature_engineering(test)

# **3. LABEL ENCODE CATEGORICALS**

In [6]:
cat_cols = ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks',
            'geo3', 'geo4', 'geo5', 'geohash']

for col in cat_cols:
    train[col] = train[col].fillna('Unknown').astype(str)
    test[col]  = test[col].fillna('Unknown').astype(str)

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0)
    le.fit(combined)
    train[col] = le.transform(train[col]).astype(int)
    test[col]  = le.transform(test[col]).astype(int)
    le_dict[col] = le

print(f"Train shape after FE: {train.shape}")


Train shape after FE: (77299, 34)


**4. DEFINE FEATURES & TARGET**

In [7]:
DROP_COLS    = ['Index', 'timestamp', 'demand']
feature_cols = [c for c in train.columns if c not in DROP_COLS]

X      = train[feature_cols].copy()
y      = train['demand'].copy()
X_test = test[feature_cols].copy()

print(f"Features ({len(feature_cols)}): {feature_cols}")


Features (31): ['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'time_minutes', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'time_sin', 'time_cos', 'geo3', 'geo4', 'geo5', 'LargeVehicles_bin', 'Landmarks_bin', 'RoadType_ord', 'Weather_ord', 'lanes_x_road', 'lanes_x_vehicles', 'temp_x_weather', 'road_x_landmark', 'is_morning_peak', 'is_evening_peak', 'is_night']


# **5. MODEL PARAMS**

In [8]:
lgb_params = {
    'objective':         'regression',
    'metric':            'rmse',
    'learning_rate':     0.05,
    'num_leaves':        127,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'n_estimators':      3000,
    'random_state':      42,
    'verbose':           -1,
}

xgb_params = {
    'objective':           'reg:squarederror',
    'learning_rate':       0.05,
    'max_depth':           7,
    'min_child_weight':    5,
    'subsample':           0.8,
    'colsample_bytree':    0.8,
    'reg_alpha':           0.1,
    'reg_lambda':          1.0,
    'n_estimators':        3000,
    'random_state':        42,
    'verbosity':           0,
    'tree_method':         'hist',
    'early_stopping_rounds': 150,  # XGBoost v3+ needs this in constructor
}

cat_params = {
    'iterations':    3000,
    'learning_rate': 0.05,
    'depth':         8,
    'l2_leaf_reg':   3,
    'loss_function': 'RMSE',
    'random_seed':   42,
    'verbose':       False,
}


# **6. 5-FOLD CV + ENSEMBLE**

In [9]:
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_lgb  = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))

print("\n" + "="*60)
print("Starting 5-Fold Cross Validation...")
print("="*60)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n── Fold {fold+1}/{N_SPLITS} ──")

    X_tr  = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_tr  = y.iloc[train_idx]
    y_val = y.iloc[val_idx]
    X_t   = X_test.copy()

    # Target encode geohash (inside fold — no leakage)
    geo_map      = pd.Series(y_tr.values, index=X_tr.index).groupby(X_tr['geohash']).mean()
    global_mean  = y_tr.mean()
    X_tr['geo_te']  = X_tr['geohash'].map(geo_map).fillna(global_mean)
    X_val['geo_te'] = X_val['geohash'].map(geo_map).fillna(global_mean)
    X_t['geo_te']   = X_t['geohash'].map(geo_map).fillna(global_mean)

    # ── LightGBM ──
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)]
    )
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    pred_lgb        += model_lgb.predict(X_t) / N_SPLITS
    print(f"  LGB  R²: {r2_score(y_val, oof_lgb[val_idx]):.5f}  (best iter: {model_lgb.best_iteration_})")

    # ── XGBoost ──
    model_xgb = xgb.XGBRegressor(**xgb_params)
    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    oof_xgb[val_idx] = model_xgb.predict(X_val)
    pred_xgb        += model_xgb.predict(X_t) / N_SPLITS
    print(f"  XGB  R²: {r2_score(y_val, oof_xgb[val_idx]):.5f}  (best iter: {model_xgb.best_iteration})")

    # ── CatBoost ──
    model_cat = CatBoostRegressor(**cat_params)
    model_cat.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=150,
    )
    oof_cat[val_idx] = model_cat.predict(X_val)
    pred_cat        += model_cat.predict(X_t) / N_SPLITS
    print(f"  CAT  R²: {r2_score(y_val, oof_cat[val_idx]):.5f}")



Starting 5-Fold Cross Validation...

── Fold 1/5 ──
  LGB  R²: 0.95319  (best iter: 1339)
  XGB  R²: 0.95402  (best iter: 1983)
  CAT  R²: 0.95416

── Fold 2/5 ──
  LGB  R²: 0.95160  (best iter: 720)
  XGB  R²: 0.95275  (best iter: 1780)
  CAT  R²: 0.95332

── Fold 3/5 ──
  LGB  R²: 0.95339  (best iter: 965)
  XGB  R²: 0.95043  (best iter: 1669)
  CAT  R²: 0.95149

── Fold 4/5 ──
  LGB  R²: 0.94839  (best iter: 1102)
  XGB  R²: 0.94783  (best iter: 1435)
  CAT  R²: 0.94664

── Fold 5/5 ──
  LGB  R²: 0.95279  (best iter: 1085)
  XGB  R²: 0.95279  (best iter: 1822)
  CAT  R²: 0.95464


# **7. OOF SCORES**

In [10]:
r2_lgb_oof = r2_score(y, oof_lgb)
r2_xgb_oof = r2_score(y, oof_xgb)
r2_cat_oof = r2_score(y, oof_cat)

print("\n" + "="*60)
print("Final OOF R² Scores:")
print(f"  LightGBM : {r2_lgb_oof:.5f}  → Score: {max(0, 100*r2_lgb_oof):.2f}")
print(f"  XGBoost  : {r2_xgb_oof:.5f}  → Score: {max(0, 100*r2_xgb_oof):.2f}")
print(f"  CatBoost : {r2_cat_oof:.5f}  → Score: {max(0, 100*r2_cat_oof):.2f}")



Final OOF R² Scores:
  LightGBM : 0.95191  → Score: 95.19
  XGBoost  : 0.95162  → Score: 95.16
  CatBoost : 0.95213  → Score: 95.21


In [11]:
best_r2      = -np.inf
best_weights = (0.34, 0.33, 0.33)

for w1 in np.arange(0.05, 0.91, 0.05):
    for w2 in np.arange(0.05, 0.91 - w1, 0.05):
        w3 = round(1.0 - w1 - w2, 4)
        if w3 <= 0:
            continue
        blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
        r2    = r2_score(y, blend)
        if r2 > best_r2:
            best_r2      = r2
            best_weights = (w1, w2, w3)

print(f"\nBest blend  → LGB: {best_weights[0]:.2f}  XGB: {best_weights[1]:.2f}  CAT: {best_weights[2]:.2f}")
print(f"Blended OOF R²  : {best_r2:.5f}")
print(f"Score (0–100)   : {max(0, 100 * best_r2):.2f}")



Best blend  → LGB: 0.35  XGB: 0.20  CAT: 0.45
Blended OOF R²  : 0.95353
Score (0–100)   : 95.35


# **8. FIND OPTIMAL BLEND WEIGHTS**

In [12]:
w1, w2, w3  = best_weights
final_preds = w1 * pred_lgb + w2 * pred_xgb + w3 * pred_cat
final_preds = np.clip(final_preds, 0, 1)

submission = pd.DataFrame({
    'Index':  test['Index'],
    'demand': final_preds
})
submission.to_csv(
    r'C:\Users\Saumya\Downloads\submission.csv',
    index=False
)
print(f"\nSubmission saved!  Shape: {submission.shape}")
print(submission.head(10))


Submission saved!  Shape: (41778, 2)
   Index    demand
0      0  0.057568
1      1  0.025906
2      2  0.037244
3      3  0.032854
4      4  0.055877
5      5  0.010817
6      6  0.034184
7      7  0.075694
8      8  0.034445
9      9  0.053608
